In [ ]:
import spacy
from collections import Counter, defaultdict
import csv
import re
import os
import pandas as pd
import numpy as np
import random

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])



/home/p318482/syntactic-bootstrapping/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Actual Substitutions and Minimal Pairs creations

In [4]:
# Regex for tokenizing words
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

In [5]:
def load_verb_bins(binned_data_dir):
    """
    Load verb bins from a nested folder structure.

    Directory structure expected:
    binned_data_dir/
        POS/
            TAG/
                bin_0.csv
                bin_1.csv
                ...

    Returns:
        dict: {(pos, tag, bin_idx): [verbs]}
    """
    verb_bins = defaultdict(list)

    for pos_folder in os.listdir(binned_data_dir):
        pos_path = os.path.join(binned_data_dir, pos_folder)
        if not os.path.isdir(pos_path):
            continue
        for tag_folder in os.listdir(pos_path):
            tag_path = os.path.join(pos_path, tag_folder)
            if not os.path.isdir(tag_path):
                continue
            for bin_file in os.listdir(tag_path):
                if not bin_file.endswith(".csv"):
                    continue
                try:
                    bin_idx = int(bin_file.split("_")[-1].split(".")[0])
                except ValueError:
                    print(f"⚠️ Skipping file (bad name): {bin_file}")
                    continue
                df = pd.read_csv(os.path.join(tag_path, bin_file))
                verbs = df["word"].dropna().astype(str).tolist()
                verb_bins[(pos_folder, tag_folder, bin_idx)].extend(verbs)

    print(f"Loaded bins for {len(verb_bins)} POS-TAG-bin combinations.")
    return verb_bins

In [ ]:
verbs_dist = load_verb_bins("./binned_data/CHILDES_rand")

verbs_dist['VERB', 'VBD', 5][:10]  # Example access

Loaded bins for 467 POS-TAG-bin combinations.


['fixed',
 'moved',
 'let',
 'pulled',
 'read',
 'spilled',
 'drew',
 'bumped',
 'slept',
 'washed']

In [ ]:
def get_verb_bin_for_verb(main_verb, verb_bins):
    """
    Return the bin key containing the original verb.
    If multiple bins contain it, pick one randomly.
    """
    pos, tag = main_verb.pos_, main_verb.tag_
    candidate_bins = [
        key for key in verb_bins.keys()
        if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
    ]
    if not candidate_bins:
        return None
    return random.choice(candidate_bins)

def generate_minimal_pairs(
    dataset_path: str,
    output_txt: str,
    verb_bins: dict,
    pattern: re.Pattern,
    spacy_model: str = "en_core_web_sm",
    min_tokens: int = 10,
    max_tokens: int = 30,
    num_replacements: int = 5
):
    """
    Generate minimal pairs for sentences in dataset_path.
    
    Each line in the output file will have: original_sentence,new_sentence

    Parameters:
    - dataset_path: path to input text file (one sentence per line)
    - output_txt: path to output TXT file
    - verb_bins: dict {(pos, tag, bin_idx): [verbs]} for sampling replacements
    - pattern: compiled regex pattern for tokenizing words
    - spacy_model: SpaCy model to use
    - min_tokens: minimum number of tokens to consider a sentence
    - num_replacements: number of minimal pair variations per sentence
    """
    
    punct_fix = re.compile(r"\s+([.,?!])")  # fix spacing before punctuation
    nlp = spacy.load(spacy_model, disable=["ner"])


    with open(dataset_path, "r", encoding="utf-8") as f, open(output_txt, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.writer(out_f, delimiter=",")
        
        for line in f:
            line = line.strip()
            if not line:
                continue

            tokens = pattern.findall(line)
            if len(tokens) < min_tokens or len(tokens) > max_tokens:
                continue  # skip short sentences

            doc = nlp(line)

            # Identify main verb (ROOT)
            main_verb = None
            for token in doc:
                if token.pos_ == "VERB" and token.dep_ == "ROOT":
                    main_verb = token
                    break
            if main_verb is None:
                continue

    
            bin_key = get_verb_bin_for_verb(main_verb, verb_bins)
            if bin_key is None:
                continue

            # Sample replacements from the same bin (exclude original)
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Replace verbs at token level
            token_texts = [t.text for t in doc]
            for rep in replacements:
                new_tokens = token_texts.copy()
                new_tokens[main_verb.i] = str(rep)
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)  # fix spacing before punctuation
                writer.writerow([line, new_sent])

    print(f"Minimal pairs saved to {output_txt}")

In [ ]:
def get_verb_bin_for_verb_stanza(main_verb, verb_bins):
        """
        Return the bin key containing the original verb.
        If multiple bins contain it, pick one randomly.
        """
        pos, tag = main_verb.upos, main_verb.xpos
        candidate_bins = [
            key for key in verb_bins.keys()
            if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
        ]
        if not candidate_bins:
            return None
        return random.choice(candidate_bins)

def generate_minimal_pairs_stanza(
    dataset_path: str,
    output_txt: str,
    verb_bins: dict,
    pattern: re.Pattern,
    nlp_stanza,
    min_tokens: int = 10,
    num_replacements: int = 5
):
    """
    Generate minimal pairs using Stanza parser.
    
    Each line in output_txt will contain: original_sentence, modified_sentence
    with standardized punctuation (no spaces before punctuation).
    """
    punct_fix = re.compile(r"\s+([.',?!])")


    with open(dataset_path, "r", encoding="utf-8") as f, open(output_txt, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.writer(out_f, delimiter=",")
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = pattern.findall(line)
            if len(tokens) < min_tokens:
                continue  # skip short sentences

            doc = nlp_stanza(line)

            # Flatten tokens
            all_words = [word for sent in doc.sentences for word in sent.words]

            # Identify main verb = root of dependency tree
            main_verb = None
            for word in all_words:
                if word.upos == "VERB" and word.deprel == 'root':  # ROOT in Stanza
                    main_verb = word
                    break
            if main_verb is None:
                continue

            # Get bin for this verb
            bin_key = get_verb_bin_for_verb_stanza(main_verb, verb_bins)
            if bin_key is None:
                continue
            if bin_key is None:
                continue

            # Sample replacement verbs
            candidates = [v for v in verb_bins[bin_key] if str(v).lower() != main_verb.text.lower()]
            if len(candidates) < num_replacements:
                replacements = candidates
            else:
                replacements = random.sample(candidates, num_replacements)

            # Replace in token list
            original_sentence = " ".join([w.text for w in all_words])
            # Apply punctuation fix to original
            original_sentence = punct_fix.sub(r"\1", original_sentence)

            for rep in replacements:
                new_tokens = []
                for word in all_words:
                    if word.id == main_verb.id:  # replace only the root verb
                        new_tokens.append(str(rep))
                    else:
                        new_tokens.append(word.text)
                # Apply punctuation fix to modified sentence
                new_sentence = " ".join(new_tokens)
                new_sentence = punct_fix.sub(r"\1", new_sentence)

                writer.writerow([original_sentence, new_sentence])

    print(f"Minimal pairs saved to {output_txt}")

## CHILDES stanza

In [ ]:
import stanza
# POS and lemma pipeline with your custom tagger model
pos_tagger_model = './syntactic-bootstrapping/saved_models/pos/en_childes_charlm_tagger.pt'

# Dependency parse pipeline with your custom parser model
parser_model = './syntactic-bootstrapping/saved_models/depparse/en_childes_charlm_parser.pt'
nlp_childes = stanza.Pipeline(
    lang='en',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=True,  # Set to True if GPU is available
    pos_model_path=pos_tagger_model,
    depparse_model_path=parser_model
)

2025-12-04 10:00:40 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-12-04 10:00:40 INFO: Downloaded file to /home/p318482/stanza_resources/resources.json
2025-12-04 10:00:40 WARNING: Language en package default expects mwt, which has been added
2025-12-04 10:00:41 INFO: Loading these models for language: en (English):
| Processor | Package                 |
---------------------------------------
| tokenize  | combined                |
| mwt       | combined                |
| pos       | /home/p318..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | /home/p318..._parser.pt |

2025-12-04 10:00:41 INFO: Using device: cuda
2025-12-04 10:00:41 INFO: Loading: tokenize
2025-12-04 10:00:44 INFO: Loading: mwt
2025-12-04 10:00:44 INFO: Loading: pos
2025-12-04 10:00:48 INFO: Loading: lemma
2025-12-04 10:00:49 INFO: Loading: deppa

In [ ]:
import re
import csv

input_file = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new/minimal_pairs_childes_rand_new.txt"
output_file = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new/minimal_pairs_childes_rand_new_correct.txt"
max_lines = 140000
# Regex to match words (simple whitespace-based tokenization)
punct_fix = re.compile(r"\s+([.',?!])")

def count_tokens(sentence):
    # Remove leading/trailing whitespace
    sentence = sentence.strip()
    # Count tokens by splitting on whitespace
    tokens = sentence.split()
    # Optionally include punctuation fixes
    tokens += punct_fix.findall(sentence)
    return len(tokens)

with open(input_file, "r", encoding="utf-8") as f_in, \
     open(output_file, "w", newline="", encoding="utf-8") as f_out:

    reader = csv.DictReader(f_in, delimiter=',')
    fieldnames = reader.fieldnames
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()

    kept = 0
    skipped = 0

    for row in reader:
        if kept >= max_lines:
            break

        s1 = row["sentence1"]
        s2 = row["sentence2"]

        if count_tokens(s1) <= 30 and count_tokens(s2) <= 30:
            writer.writerow(row)
            kept += 1
        else:
            skipped += 1

print(f"Done! Kept {kept} rows, skipped {skipped} rows.")

Done! Kept 140000 rows, skipped 2291 rows.


## CHILDES stanza rand

In [ ]:
binned_data_dir = "./syntactic-bootstrapping/binned_data/CHILDES_rand_new_new"
verb_bins = load_verb_bins(binned_data_dir)


dataset_path = "./syntactic-bootstrapping/corpora/CHILDES_rand_new_new/original/childes_rand_new_new.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new.txt"

pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

generate_minimal_pairs_stanza(dataset_path, output_txt, verb_bins, pattern, nlp_childes)

Loaded bins for 467 POS-TAG-bin combinations.
Minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new.txt


## WIKI

In [ ]:
binned_data_dir = './syntactic-bootstrapping/binned_data/WIKI'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./syntactic-bootstrapping/corpora/WIKI/original/wiki.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)


Loaded bins for 436 POS-TAG-bin combinations.
Minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki.txt


## WIKI rand

In [ ]:
binned_data_dir = './syntactic-bootstrapping/binned_data/WIKI_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./syntactic-bootstrapping/corpora/WIKI_rand/original/wiki_rand.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI_rand/minimal_pairs_wiki_rand.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)

Loaded bins for 436 POS-TAG-bin combinations.
Minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI_rand/minimal_pairs_wiki_rand.txt


## BNC

In [ ]:
binned_data_dir = './syntactic-bootstrapping/binned_data/BNC'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./syntactic-bootstrapping/corpora/BNC/original/bnc.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/minimal_pairs_bnc.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)


Loaded bins for 408 POS-TAG-bin combinations.
Minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/minimal_pairs_bnc.txt


In [ ]:
binned_data_dir = './syntactic-bootstrapping/binned_data/BNC_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./syntactic-bootstrapping/corpora/BNC_rand/original/bnc_rand.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/minimal_pairs_bnc_rand.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)

Loaded bins for 414 POS-TAG-bin combinations.
Minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/minimal_pairs_bnc_rand.txt


## BNC_SPOKEN

In [ ]:
binned_data_dir = './syntactic-bootstrapping/binned_data/BNC_SPOKEN_CANDOR_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./syntactic-bootstrapping/corpora/CANDOR_rand/original/candor.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CANDOR_rand/minimal_pairs_candor_new_again.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)


## Clean Minimal Pairs CHILDES from wan na occurrences

In [ ]:
import pandas as pd

# Load your dataset
df = pd.read_csv("./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new.txt")  # or pd.read_excel(...)

# Use regex to filter out rows where 'sentence1' contains 'wan na'
# The regex r'\bwan\s+na\b' ensures we match 'wan na' as separate words
df_cleaned = df[~df['sentence1'].str.contains(r'\bwan\s+na\b', regex=True)]
df_cleaned = df[~df['sentence1'].str.contains(r'\bgot\s+ta\b', regex=True)]
df_cleaned = df[~df['sentence1'].str.contains(r'\bgon\s+na\b', regex=True)]

# Save the cleaned dataset
df_cleaned.to_csv("./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new_cleaned.txt", index=False)


## After the creation of the minimal pair dataset we have to go through other steps of cleaning and checks 

## POST-HOC ANALYSIS 
What is the bin frequency distribution of the target verbs in the 10000 semantic minimal pairs I evaluate?

In [ ]:
import csv
import re
import pandas as pd
import os
import stanza
import spacy
from collections import Counter,defaultdict


auxiliaries = [
    "am", "are", "is", "was", "were",
    "have", "has", "had",
    "do", "does", "did",
    "can", "could",
    "will", "would",
    "shall", "should",
    "may", "might",
    "must"
]

INSIDE_BRACKETS = re.compile(r"([\(\[\{])\s*(.*?)\s*([\)\]\}])")
INSIDE_QUOTES = re.compile(r"([\"'‘“”])\s*(.*?)\s*([\"'“’”])")
REMOVE_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.:;!?])")          # space before punctuation
punct_fix = re.compile(r"\s+([,.'?!])$")      # space before . ? !
ellipsis_fix = re.compile(r"\s+(\.\.\.)$")  # space before ...
underscores_fix = re.compile(r"_([A-Za-z0-9]+)_")  # remove surrounding underscores
aux_pattern = re.compile(r"\b(" + "|".join(auxiliaries) + r")\s+n['’]t\b", flags=re.IGNORECASE)
generic_clitic_pattern = re.compile(
    r"\b([A-Za-z]+)\s+('re|'m|'s|'ve|'d|'ll|n['’]t)\b", flags=re.IGNORECASE
)
# common split-to-joined contractions
split_to_joined = {
    r"\bwan\s+na\b": "wanna",
    r"\bgon\s+na\b": "gonna",
    r"\bgot\s+ta\b": "gotta",
    r"\blem\s+me\b": "lemme",
    r"\bgim\s+me\b": "gimme",
    r"\dun\s+no\s+know\b": "dunno",
    r"\bought\s+to\b": "oughta",
    r"\bcan\s+not\b": "cannot",
}
split_patterns = [(re.compile(k, flags=re.IGNORECASE), v) for k, v in split_to_joined.items()]
hyphen_fix_pattern = re.compile(r"\b([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9]+)\b")



def clean_sentence(sentence: str) -> str:
    """Apply all cleaning rules to a single sentence."""
    sentence = sentence.strip().lower()
    sentence = punct_fix.sub(r"\1", sentence)
    sentence = ellipsis_fix.sub(r"\1", sentence)
    sentence = underscores_fix.sub(r"\1", sentence)
    sentence = aux_pattern.sub(lambda m: m.group(1).lower() + "n't", sentence)
    sentence = generic_clitic_pattern.sub(lambda m: m.group(1) + m.group(2), sentence)
    for pattern, replacement in split_patterns:
        sentence = pattern.sub(replacement, sentence)
    sentence = hyphen_fix_pattern.sub(r"\1-\2", sentence)
    if sentence.startswith("' "):
        sentence = sentence.replace("' ", "'", 1)
    if sentence.startswith('" '):
        sentence = sentence.replace('" ', '"', 1)
    if sentence.endswith('  . "'):
        sentence = sentence.replace('  . "', '."', 1)
    if ' ...' in sentence: 
        sentence = sentence.replace(' ...', '...', 1)
    if ' . ..' in sentence:
        sentence = sentence.replace(' . ..', '...', 1)
    if ' . . .' in sentence:
        sentence = sentence.replace(' . . .', '...', 1)
    sentence = REMOVE_SPACE_BEFORE_PUNCT.sub(r"\1", sentence)
    sentence = INSIDE_BRACKETS.sub(r"\1\2\3", sentence)
    sentence = INSIDE_QUOTES.sub(r"\1\2\3", sentence)
    sentence = re.sub(r"  +", " ", sentence)
    return sentence

def clean_minimal_pairs_csv(datasets: dict, output_dir: str):
    """Apply cleaning to both columns of minimal pairs CSV files."""
    os.makedirs(output_dir, exist_ok=True)

    for split, input_file in datasets.items():
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        output_file = os.path.join(output_dir, f"{base_name}.{split}.cleaned.csv")
        print(f"Processing {split} split for minimal pairs cleaning...")

        with open(input_file, "r", encoding="utf-8") as f_in, \
             open(output_file, "w", encoding="utf-8", newline="") as f_out:

            reader = csv.reader(f_in)
            writer = csv.writer(f_out)
            header = next(reader)  # keep header
            writer.writerow(header)

            for row in reader:
                if len(row) < 2:
                    continue
                s1, s2 = row[0], row[1]
                cleaned_s1 = clean_sentence(s1)
                cleaned_s2 = clean_sentence(s2)
                writer.writerow([cleaned_s1, cleaned_s2])

        print(f"Saved cleaned {split} to {output_file}")


In [ ]:
datasets = {
    "childes": "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new_cleaned.txt"
}

output_dir = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "candor": "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor.txt"
}

output_dir = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "bnc": "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/minimal_pairs_bnc_rand.txt"
}

output_dir = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "wiki": "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/minimal_pairs_wiki_rand.txt"
}

output_dir = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
childes = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/cleaned/minimal_pairs_childes_rand_new_cleaned.childes.cleaned.csv"
wiki = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand.wiki.cleaned.csv"
bnc = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand.bnc.cleaned.csv"
candor = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/cleaned/minimal_pairs_candor.candor.cleaned.csv"

In [ ]:
def normalize_sentence_for_comparison(sentence: str) -> str:
    """
    Lowercase and remove/normalize spaces and punctuation for token alignment.
    Keeps only word characters and single spaces.
    """
    sentence = sentence.lower().strip()
    # replace punctuation with spaces
    sentence = re.sub(r"[^\w\s']", " ", sentence)
    # collapse multiple spaces
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence

def extract_target_verbs_with_pos_spacy_normalized(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group),
    along with its lemma, coarse POS, and fine-grained POS using SpaCy.
    Normalizes sentences to avoid mismatches due to spaces/punctuation.
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)

            # normalize sentences for token comparison
            s1_norm = normalize_sentence_for_comparison(s1)
            s2_norm = normalize_sentence_for_comparison(s2)

            # tokenize original sentences with SpaCy
            doc1 = nlp(s1_norm)
            doc2 = nlp(s2_norm)

            # flatten tokens
            tokens1 = [t for t in doc1]
            tokens2 = [t for t in doc2]

            # find first mismatch using normalized tokens
            for tok1, tok2 in zip(tokens1, tokens2):
                tok1_norm = normalize_sentence_for_comparison(tok1.text)
                tok2_norm = normalize_sentence_for_comparison(tok2.text)
                if tok1_norm != tok2_norm:
                    results.append({
                        'sentence1': s1,
                        'verb': tok1.text,
                        'lemma': tok1.lemma_,
                        'upos': tok1.pos_,
                        'xpos': tok1.tag_
                    })
                    break

    return results


In [ ]:
def extract_target_verbs_with_pos_spacy(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group),
    along with its lemma, coarse POS, and fine-grained POS using SpaCy.
    
    Args:
        path: path to CSV file containing minimal pairs (sentence1, sentence2)
        nlp: SpaCy NLP pipeline with POS tagging
        max_lines: max number of lines to process
    Returns:
        List of dicts: [{'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)

            # tokenize with SpaCy
            doc1 = nlp(s1)
            doc2 = nlp(s2)

            # iterate over tokens and find first mismatch
            for tok1, tok2 in zip(doc1, doc2):
                if tok1.text != tok2.text:
                    results.append({
                        'sentence1': s1,
                        'verb': tok1.text,
                        'lemma': tok1.lemma_,
                        'upos': tok1.pos_,
                        'xpos': tok1.tag_
                    })
                    break

    return results

In [ ]:
def extract_target_verbs_with_pos_stanza(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group), 
    along with its lemma, coarse POS (UPOS), and fine-grained POS (XPOS).
    
    Args:
        path: path to CSV file containing minimal pairs (sentence1, sentence2)
        nlp: stanza NLP pipeline with POS and depparse
        max_lines: max number of lines to process
    Returns:
        List of dicts: [{'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)


            # tokenize with stanza
            doc1 = nlp(s1)
            doc2 = nlp(s2)

            # flatten tokens (stanza outputs sentences)
            tokens1 = [t for sent in doc1.sentences for t in sent.tokens]
            tokens2 = [t for sent in doc2.sentences for t in sent.tokens]

            # iterate over tokens and find first mismatch
            for tok1, tok2 in zip(tokens1, tokens2):
                if tok1.text != tok2.text:
                    # take the first word of the token (usually just one)
                    word = tok1.words[0]
                    results.append({
                        'sentence1': s1,
                        'lemma': word.lemma,
                        'verb': word.text,
                        'upos': word.upos,
                        'xpos': word.xpos
                    })
                    break

    return results


In [ ]:
def load_binned_words(binned_dir):
    """
    Load all words from the binned CSVs into a nested dictionary:
    binned_words[coarse_pos][fine_tag][bin_idx] = list of words
    """
    binned_words = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for pos in os.listdir(binned_dir):
        pos_path = os.path.join(binned_dir, pos)
        if not os.path.isdir(pos_path):
            continue
        for tag in os.listdir(pos_path):
            tag_path = os.path.join(pos_path, tag)
            if not os.path.isdir(tag_path):
                continue
            for bin_file in os.listdir(tag_path):
                if bin_file.endswith(".csv"):
                    bin_idx = int(bin_file.split("_")[1].split(".")[0])
                    df = pd.read_csv(os.path.join(tag_path, bin_file))
                    words = df["word"].tolist()
                    binned_words[pos][tag][bin_idx].extend(words)

    return binned_words

In [ ]:
def count_target_verbs_in_bins(target_verbs_info, binned_words):
    """
    Count how many target verbs fall into each bin (1-10).

    Args:
        target_verbs_info: list of dicts with keys: 'verb', 'lemma', 'upos', 'xpos'
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words

    Returns:
        dict {bin_1: count, bin_2: count, ..., bin_10: count}
    """
    bin_counts = defaultdict(int)

    for v in target_verbs_info:
        verb = v['verb']
        upos = v['upos']
        xpos = v['xpos']

        # check which bin the verb falls into
        found = False
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_counts[f'bin_{bin_idx}'] += 1
                    found = True
                    break  # assume each verb belongs to only one bin

        if not found:
            bin_counts['bin_not_found'] += 1  # optional: track verbs not in any bin

    # ensure all bins 1-10 exist (even if 0)
    for i in range(1, 11):
        bin_counts.setdefault(f'bin_{i}', 0)

    return dict(bin_counts)

In [ ]:
def count_target_verbs_in_bins_with_not_found(target_verbs_info, binned_words):
    """
    Count how many target verbs fall into each bin (1-10) and save the verbs
    that were not found in any bin.

    Args:
        target_verbs_info: list of dicts with keys: 'verb', 'lemma', 'upos', 'xpos'
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words

    Returns:
        bin_counts: dict {bin_1: count, ..., bin_10: count, bin_not_found: count}
        not_found_verbs: list of dicts [{'sentence1':..., 'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    bin_counts = defaultdict(int)
    not_found_verbs = []

    for v in target_verbs_info:
        verb = v['verb']
        upos = v['upos']
        xpos = v['xpos']

        found = False
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_counts[f'bin_{bin_idx}'] += 1
                    found = True
                    break

        if not found:
            bin_counts['bin_not_found'] += 1
            not_found_verbs.append(v)

    # ensure all bins 1-10 exist
    for i in range(1, 11):
        bin_counts.setdefault(f'bin_{i}', 0)

    return dict(bin_counts), not_found_verbs


## CHILDES 

In [ ]:
pos_tagger_model = './syntactic-bootstrapping/saved_models/pos/en_childes_charlm_tagger.pt'
parser_model = './syntactic-bootstrapping/saved_models/depparse/en_childes_charlm_parser.pt'


nlp_childes = stanza.Pipeline(
    lang='en',
     processors='tokenize,pos,lemma,depparse',
     use_gpu=True,
     pos_model_path=pos_tagger_model,
     depparse_model_path=parser_model
 )

bins_childes = load_binned_words('./syntactic-bootstrapping/binned_data/CHILDES_rand_new_new')
childes_verbs_info = extract_target_verbs_with_pos_stanza("./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/cleaned/minimal_pairs_childes_rand_new_cleaned.childes.cleaned.csv", nlp_childes)
bin_counts_childes, not_found_verbs = count_target_verbs_in_bins_with_not_found(childes_verbs_info, bins_childes)
print("Bin counts:", bin_counts_childes)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("childes_bin_not_found.csv", index=False)

## WIKIPEDIA

In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")
bins_wiki = load_binned_words('./syntactic-bootstrapping/binned_data/WIKI_rand')
wiki_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand.wiki.cleaned.csv', nlp_spacy)
bin_counts_wiki, not_found_verbs = count_target_verbs_in_bins_with_not_found(wiki_verbs_info, bins_wiki)
print("Bin counts:", bin_counts_wiki)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("wiki_bin_not_found.csv", index=False)

## BNC

In [ ]:
bins_bnc= load_binned_words('./syntactic-bootstrapping/binned_data/BNC_rand')
bnc_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand.bnc.cleaned.csv', nlp_spacy)
bin_counts_bnc, not_found_verbs = count_target_verbs_in_bins_with_not_found(bnc_verbs_info, bins_bnc)
print("Bin counts:", bin_counts_bnc)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("bnc_bin_not_found.csv", index=False)

## CANDOR

In [ ]:
bins_candor = load_binned_words('./syntactic-bootstrapping/corpora/CANDOR_rand')
candor_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CANDOR_rand/cleaned/minimal_pairs_candor_new.candor.cleaned.csv', nlp_spacy, max_lines=140000)
bin_counts_candor, not_found_verbs = count_target_verbs_in_bins_with_not_found(candor_verbs_info, bins_candor)
print("Bin counts:", bin_counts_candor)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("candor_bin_not_found.csv", index=False)

## CREATE A NEW DATASET of minimal pairs where you control also for frequency bins of the verbs

In [ ]:
def create_minimal_pair_with_bin(input_csv, output_csv, target_verbs_info, binned_words, max_lines=140000):
    """
    Create a new minimal pair CSV with an extra column 'bin_number' indicating
    the bin of the target verb in sentence1.
    
    Args:
        input_csv: path to original minimal pairs CSV (sentence1, sentence2)
        output_csv: path to save new CSV (sentence1, sentence2, bin_number)
        target_verbs_info: list of dicts [{'sentence1':..., 'verb':..., 'lemma':..., 'upos':..., 'xpos':...}]
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words
        max_lines: max lines to process
    """

    # Build a mapping from sentence1 to its bin_number
    s1_to_bin = {}
    for info in target_verbs_info:
        s1 = info['sentence1']
        if s1 in s1_to_bin:
            continue  # already assigned
        verb = info['verb']
        upos = info['upos']
        xpos = info['xpos']
        bin_number = 'bin_not_found'
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_number = f'bin_{bin_idx}'
                    break
        s1_to_bin[s1] = bin_number

    # Read original CSV and write new CSV with bin_number
    with open(input_csv, "r", encoding="utf-8") as fin, \
         open(output_csv, "w", encoding="utf-8", newline='') as fout:
        reader = csv.reader(fin)
        writer = csv.writer(fout)
        header = next(reader)
        writer.writerow(header + ['bin_number'])

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue
            s1, s2 = row[0].strip(), row[1].strip()
            bin_number = s1_to_bin.get(s1, 'bin_not_found')
            writer.writerow([s1, s2, bin_number])

    print(f"Saved minimal pair CSV with bin numbers to: {output_csv}")


In [ ]:
create_minimal_pair_with_bin(
    input_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/cleaned/childes_cleaned_2.csv",
    output_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/cleaned/childes_cleaned_2_with_bins.csv",
    target_verbs_info=childes_verbs_info,
    binned_words=bins_childes,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand.bnc.cleaned.csv",
    output_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand_with_bins.csv",
    target_verbs_info=bnc_verbs_info,
    binned_words=bins_bnc,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand.wiki.cleaned.csv",
    output_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand_with_bins.csv",
    target_verbs_info=wiki_verbs_info,
    binned_words=bins_wiki,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/cleaned/minimal_pairs_candor.candor.cleaned.csv",
    output_csv="./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/cleaned/minimal_pairs_candor_with_bins.csv",
    target_verbs_info=candor_verbs_info,
    binned_words=bins_candor,
    max_lines=140000
)

## PLOT THE DISTRIBUTION

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

bin_counts_childes = {'bin_8': 4422, 
                      'bin_5': 1588, 
                      'bin_7': 4161, 
                      'bin_9': 15070, 
                      'bin_4': 747, 
                      'bin_2': 278, 
                      'bin_6': 2180, 
                    #'bin_not_found': 46,
                    'bin_3': 494, 
                    'bin_0': 110, 
                    'bin_1': 125, 
                    'bin_10': 0}


bin_counts_wiki = {'bin_7': 7070, 
                   'bin_5': 3946, 
                   'bin_6': 5862, 
                   'bin_9': 4261, 
                   'bin_4': 2202, 
                   'bin_1': 371, 
                   'bin_8': 3755, 
                   'bin_3': 1295, 
                   'bin_2': 617, 
                   'bin_0': 238, 
                   #'bin_not_found': 20, 
                   'bin_10': 0}

bin_counts_bnc = {'bin_3': 1496, 
                  'bin_5': 3718, 
                  'bin_6': 5456, 
                  'bin_4': 2361, 
                  'bin_1': 390, 
                  'bin_8': 5296, 
                  'bin_2': 821, 
                  'bin_7': 5630, 
                  'bin_9': 4805, 
                  'bin_0': 252, 
                  #'bin_not_found': 40, 
                  'bin_10': 0}

bin_counts_candor = {'bin_7': 4202, 
                  'bin_6': 3493, 
                  'bin_8': 5916, 
                  'bin_9': 13032, 
                  'bin_5': 2025, 
                  'bin_3': 800, 
                  'bin_4': 1286, 
                  'bin_2': 468, 
                  'bin_0': 206, 
                  'bin_1': 287, 
                  #'bin_not_found': 16, 
                  'bin_10': 0}

# Combine into DataFrame
df = pd.DataFrame({
    'Childes': bin_counts_childes,
    'Wiki': bin_counts_wiki,
    'BNC': bin_counts_bnc,
    'Candor': bin_counts_candor
})
df = df.sort_index(key=lambda x: [int(b.split('_')[1]) for b in x])

domains = ['Childes', 'Wiki', 'BNC', 'Candor']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']

n_bins = len(df)
n_domains = len(domains)
bar_width = 0.2

# Create figure
fig, axes = plt.subplots(nrows=1, ncols=n_bins, figsize=(30, 5), sharey=True)
axes = axes.flatten()

for i, bin_name in enumerate(df.index):
    freqs = df.loc[bin_name, domains].values
    # center the bars: shift each bar so they touch each other
    offsets = np.arange(n_domains) * bar_width
    axes[i].bar(offsets, freqs, width=bar_width, color=colors)
    
    # set x-ticks in the middle of the group of bars
    axes[i].set_xticks(offsets)
    axes[i].set_xticklabels(domains, rotation=45, fontsize=12)
    axes[i].set_title(bin_name, fontsize=16)

axes[0].set_ylabel('Frequency', fontsize=16)

plt.tight_layout()
plt.show()


## Generate physical and mental minimal pairs

In [6]:
def generate_minimal_pairs_split(
    dataset_path: str,
    output_physical: str,
    output_mental: str,
    verb_bins: dict,
    pattern: re.Pattern,
    spacy_model: str = "en_core_web_sm",
    min_tokens: int = 10,
    max_tokens: int = 30,
    num_replacements: int = 5
):
    """
    Generate minimal pairs for sentences, replacing main verbs with others from the same bin.
    Allows any inflected form of physical and mental verbs.
    """
    punct_fix = re.compile(r"\s+([.,?!])")
    physical = ['take', 'say', 'come', 'play', 'push', 'sit', 'pull', 'eat', 'make', 'call', 'catch', 'put', 'find']
    mental   = ['see', 'look', 'want', 'know', 'like', 'think']

    nlp = spacy.load(spacy_model, disable=["ner"])

    with open(dataset_path, "r", encoding="utf-8") as f, \
         open(output_physical, "w", encoding="utf-8") as out_phys, \
         open(output_mental, "w", encoding="utf-8") as out_ment:

        for line in f:
            line = line.strip()
            if not line:
                continue

            tokens_in_line = pattern.findall(line)
            if len(tokens_in_line) < min_tokens or len(tokens_in_line) > max_tokens:
                continue

            doc = nlp(line)

            # Find main verb (ROOT) whose lemma is in physical or mental lists
            main_verb = None
            for token in doc:
                if token.pos_ == "VERB" and token.dep_ == "ROOT":
                    if token.lemma_.lower() in physical + mental:
                        main_verb = token
                        break
            if main_verb is None:
                continue

            pos, tag = main_verb.pos_, main_verb.tag_

            # Find bin(s) containing the original verb (matching the exact form)
            candidate_bins = [
                key for key in verb_bins.keys()
                if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
            ]
            if not candidate_bins:
                continue

            bin_key = random.choice(candidate_bins)

            # Sample replacements from the same bin, excluding the original verb
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Determine output writer
            writer = out_phys if main_verb.lemma_.lower() in physical else out_ment

            # Fix punctuation for original sentence
            original_sent = punct_fix.sub(r"\1", line)

            # Token-level replacement
            token_texts = [t.text for t in doc]
            for rep in replacements:
                new_tokens = token_texts.copy()
                new_tokens[main_verb.i] = rep
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                writer.write(f"{original_sent},{new_sent}\n")

    print(f"Physical minimal pairs saved to {output_physical}")
    print(f"Mental minimal pairs saved to {output_mental}")

In [7]:
def generate_minimal_pairs_split_stanza(
    dataset_path: str,
    output_physical: str,
    output_mental: str,
    verb_bins: dict,
    pattern: re.Pattern,
    pos_model_path: str,
    parser_model_path: str,
    min_tokens: int = 10,
    num_replacements: int = 5,
    use_gpu: bool = True
):

    punct_fix = re.compile(r"\s+([.,?!])")
    physical = ['take', 'say', 'come', 'play', 'push', 'sit', 'pull', 'eat', 'make', 'call', 'catch', 'put', 'find']
    mental   = ['see', 'look', 'want', 'know', 'like', 'think']

    # Initialize Stanza pipeline
    nlp = stanza.Pipeline(
        lang='en',
        processors='tokenize,pos,lemma,depparse',
        use_gpu=use_gpu,
        pos_model_path=pos_model_path,
        depparse_model_path=parser_model_path,
        tokenize_no_ssplit=True
    )

    def get_bin_for_verb(main_verb):
        """
        Returns a bin key containing the main verb (exact form),
        ensuring replacements come from the same frequency bin.
        """
        pos, tag = main_verb.upos, main_verb.xpos
        candidate_bins = [
            key for key in verb_bins.keys()
            if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
        ]
        if not candidate_bins:
            return None
        return random.choice(candidate_bins)

    with open(dataset_path, "r", encoding="utf-8") as f, \
         open(output_physical, "w", encoding="utf-8") as out_phys, \
         open(output_mental, "w", encoding="utf-8") as out_ment:

        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens_in_line = pattern.findall(line)
            if len(tokens_in_line) < min_tokens:
                continue

            doc = nlp(line)
            main_verb = None

            # Find root verb whose lemma is in physical or mental
            for sent in doc.sentences:
                for word in sent.words:
                    if word.upos == "VERB" and word.deprel == "root":
                        if word.lemma.lower() in physical + mental:
                            main_verb = word
                            break
                if main_verb:
                    break
            if main_verb is None:
                continue

            # Determine bin for original verb
            bin_key = get_bin_for_verb(main_verb)
            if bin_key is None:
                continue

            # Sample replacements from same bin, excluding original verb
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Determine output writer
            writer = out_phys if main_verb.lemma.lower() in physical else out_ment

            # Original sentence
            original_sent = " ".join([w.text for w in doc.sentences[0].words])
            original_sent = punct_fix.sub(r"\1", original_sent)

            # Replacement sentences (token-level)
            for rep in replacements:
                new_tokens = [w.text for w in doc.sentences[0].words]
                new_tokens[main_verb.id - 1] = str(rep)  # Stanza IDs are 1-based
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                writer.write(f"{original_sent},{new_sent}\n")

    print(f"Physical minimal pairs saved to {output_physical}")
    print(f"Mental minimal pairs saved to {output_mental}")


In [8]:
def generate_min_pairs_caus_noncaus(
    dataset_path: str,
    output_causative: str,
    output_non_causative: str,
    verb_bins: dict,
    pattern: re.Pattern,
    spacy_model: str = "en_core_web_sm",
    min_tokens: int = 10,
    max_tokens: int = 30,
    num_replacements: int = 5
):
    """
    Generate minimal pairs for sentences, replacing main verbs with others from the same bin.
    Allows any inflected form of physical and mental verbs.
    """
    punct_fix = re.compile(r"\s+([.,?!])")
    causatives = ['begin', 'boil', 'break', 'burn', 'change', 'close', 'destroy', 'dry', 'fill', 'freeze', 'gather', 'kill', 'lose', "melt", "open", "raise", "roll", "sink", "spread", "stop", "teach", "turn"]
    non_causatives   = ['die', 'go', 'look', 'talk', 'want', 'like', 'think', "say", "take"]

    nlp = spacy.load(spacy_model, disable=["ner"])

    with open(dataset_path, "r", encoding="utf-8") as f, \
         open(output_causative, "w", encoding="utf-8") as out_caus, \
         open(output_non_causative, "w", encoding="utf-8") as out_noncaus:

        for line in f:
            line = line.strip()
            if not line:
                continue

            tokens_in_line = pattern.findall(line)
            if len(tokens_in_line) < min_tokens or len(tokens_in_line) > max_tokens:
                continue

            doc = nlp(line)

            # Find main verb (ROOT) whose lemma is in physical or mental lists
            main_verb = None
            for token in doc:
                if token.pos_ == "VERB" and token.dep_ == "ROOT":
                    if token.lemma_.lower() in causatives + non_causatives:
                        main_verb = token
                        break
            if main_verb is None:
                continue


            bin_key = get_verb_bin_for_verb(main_verb, verb_bins)
            if bin_key is None:
                continue

            # Sample replacements from the same bin, excluding the original verb
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Determine output writer
            writer = out_caus if main_verb.lemma_.lower() in causatives else out_noncaus

            # Fix punctuation for original sentence
            original_sent = punct_fix.sub(r"\1", line)

            # Token-level replacement
            token_texts = [t.text for t in doc]
            for rep in replacements:
                new_tokens = token_texts.copy()
                new_tokens[main_verb.i] = rep
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                writer.write(f"{original_sent},{new_sent}\n")

    print(f"Physical minimal pairs saved to {out_caus}")
    print(f"Mental minimal pairs saved to {out_noncaus}")

In [9]:
def generate_min_pairs_caus_noncaus_stanza(
    dataset_path: str,
    output_causative: str,
    output_noncausative: str,
    verb_bins: dict,
    pattern: re.Pattern,
    pos_model_path: str,
    parser_model_path: str,
    min_tokens: int = 10,
    num_replacements: int = 5,
    use_gpu: bool = True
):

    punct_fix = re.compile(r"\s+([.,?!])")
    causatives = ['begin', 'boil', 'break', 'burn', 'change', 'close', 'destroy', 'dry', 'fill', 'freeze', 'gather', 'kill', 'lose', "melt", "open", "raise", "roll", "sink", "spread", "stop", "teach", "turn"]
    non_causatives   = ['die', 'go', 'look', 'talk', 'want', 'like', 'think', "say", "take"]

    # Initialize Stanza pipeline
    nlp = stanza.Pipeline(
        lang='en',
        processors='tokenize,pos,lemma,depparse',
        use_gpu=use_gpu,
        pos_model_path=pos_model_path,
        depparse_model_path=parser_model_path,
        tokenize_no_ssplit=True
    )

    with open(dataset_path, "r", encoding="utf-8") as f, \
         open(output_causative, "w", encoding="utf-8") as out_caus, \
         open(output_noncausative, "w", encoding="utf-8") as out_noncaus:

        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens_in_line = pattern.findall(line)
            if len(tokens_in_line) < min_tokens:
                continue

            doc = nlp(line)
            main_verb = None

            # Find root verb whose lemma is in physical or mental
            for sent in doc.sentences:
                for word in sent.words:
                    if word.upos == "VERB" and word.deprel == "root":
                        if word.lemma.lower() in causatives + non_causatives:
                            main_verb = word
                            break
                if main_verb:
                    break
            if main_verb is None:
                continue

            # Determine bin for original verb
            bin_key = get_verb_bin_for_verb_stanza(main_verb, verb_bins)
            if bin_key is None:
                continue

            # Sample replacements from same bin, excluding original verb
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Determine output writer
            writer = out_caus if main_verb.lemma.lower() in causatives else out_noncaus

            # Original sentence
            original_sent = " ".join([w.text for w in doc.sentences[0].words])
            original_sent = punct_fix.sub(r"\1", original_sent)

            # Replacement sentences (token-level)
            for rep in replacements:
                new_tokens = [w.text for w in doc.sentences[0].words]
                new_tokens[main_verb.id - 1] = str(rep)  # Stanza IDs are 1-based
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                writer.write(f"{original_sent},{new_sent}\n")

    print(f"Physical minimal pairs saved to {out_caus}")
    print(f"Mental minimal pairs saved to {out_noncaus}")


## CHILDES spacy

In [ ]:
dataset_path = "./syntactic-bootstrapping/corpora/CHILDES/original/childes.test.txt"   # input dataset (one sentence per line)
output_physical = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_spacy/minimal_pairs_childes_physical_spacy.txt"
output_mental   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_spacy/minimal_pairs_childes_mental_spacy.txt"
binned_dir = "./syntactic-bootstrapping/binned_data/CHILDES_spacy"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_minimal_pairs_split(
    dataset_path,
    output_physical,
    output_mental,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    num_replacements=5
)

Loaded bins for 500 POS-TAG-bin combinations.
Physical minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_spacy/minimal_pairs_childes_physical_spacy.txt
Mental minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_spacy/minimal_pairs_childes_mental_spacy.txt


## CHILDES stanza

In [ ]:
dataset_path = "./syntactic-bootstrapping/corpora/CHILDES/original/childes.test.txt"   # input dataset (one sentence per line)
output_physical = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_physical_stanza.txt"
output_mental   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_mental_stanza.txt"
binned_dir = "./syntactic-bootstrapping/binned_data/CHILDES_stanza"

pos_tagger_model = './syntactic-bootstrapping/saved_models/pos/en_childes_charlm_tagger.pt'
parser_model = './syntactic-bootstrapping/saved_models/depparse/en_childes_charlm_parser.pt'

verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

generate_minimal_pairs_split_stanza(
    dataset_path,
    output_physical,
    output_mental,
    verb_bins,
    pattern,
    pos_tagger_model,
    parser_model,
    min_tokens=10,
    num_replacements=5,
    use_gpu=True
)

2025-09-23 13:45:23 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Loaded bins for 470 POS-TAG-bin combinations.


2025-09-23 13:45:23 INFO: Downloaded file to /home/p318482/stanza_resources/resources.json
2025-09-23 13:45:23 WARNING: Language en package default expects mwt, which has been added
2025-09-23 13:45:24 INFO: Loading these models for language: en (English):
| Processor | Package                 |
---------------------------------------
| tokenize  | combined                |
| mwt       | combined                |
| pos       | /home/p318..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | /home/p318..._parser.pt |

2025-09-23 13:45:24 INFO: Using device: cuda
2025-09-23 13:45:24 INFO: Loading: tokenize
2025-09-23 13:45:24 INFO: Loading: mwt
2025-09-23 13:45:24 INFO: Loading: pos
2025-09-23 13:45:28 INFO: Loading: lemma
2025-09-23 13:45:29 INFO: Loading: depparse
2025-09-23 13:45:30 INFO: Done loading processors!


Physical minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_physical_stanza.txt
Mental minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_mental_stanza.txt


In [ ]:
dataset_path = "./Desktop/syntactic-bootstrapping/corpora/CHILDES/original/childes.test.txt"   # input dataset (one sentence per line)
output_causative = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_causatives.txt"
output_non_causative   = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_non_causatives.txt"
binned_dir = "./Desktop/syntactic-bootstrapping/binned_data/CHILDES_stanza"

pos_tagger_model = './Desktop/stanza/saved_models/pos/en_childes_charlm_tagger.pt'
parser_model = './Desktop/stanza/saved_models/depparse/en_childes_charlm_parser.pt'

verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

generate_min_pairs_caus_noncaus_stanza(
    dataset_path,
    output_causative,
    output_non_causative,
    verb_bins,
    pattern,
    pos_tagger_model,
    parser_model,
    min_tokens=10,
    num_replacements=5,
    use_gpu=True
)

## WIKIPEDIA

In [ ]:
dataset_path = "./syntactic-bootstrapping/corpora/WIKI/original/wiki.test.txt"   # input dataset (one sentence per line)
output_physical = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_physical.txt"
output_mental   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_mental.txt"
binned_dir = "./syntactic-bootstrapping/binned_data/WIKI"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_minimal_pairs_split(
    dataset_path,
    output_physical,
    output_mental,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    num_replacements=5
)

In [ ]:
dataset_path = "./Desktop/syntactic-bootstrapping/corpora/WIKI/original/wiki.test.txt"   # input dataset (one sentence per line)
output_causatives = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_causatives.txt"
output_non_causatives   = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_non_causatives.txt"
binned_dir = "./Desktop/syntactic-bootstrapping/binned_data/WIKI"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_min_pairs_caus_noncaus(
    dataset_path,
    output_causatives,
    output_non_causatives,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    max_tokens=30,
    num_replacements=5
)

## BNC WRITTEN

In [ ]:
dataset_path = "./syntactic-bootstrapping/corpora/BNC/original/bnc.test.txt"   # input dataset (one sentence per line)
output_physical = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_physical.txt"
output_mental   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_mental.txt"
binned_dir = "./syntactic-bootstrapping/binned_data/BNC"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_minimal_pairs_split(
    dataset_path,
    output_physical,
    output_mental,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    num_replacements=5)

In [ ]:
dataset_path = "./Desktop/syntactic-bootstrapping/corpora/BNC/original/bnc.test.txt"   # input dataset (one sentence per line)
output_causatives = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_causatives.txt"
output_non_causatives   = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_non_causatives.txt"
binned_dir = "./Desktop/syntactic-bootstrapping/binned_data/BNC"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_min_pairs_caus_noncaus(
    dataset_path,
    output_causatives,
    output_non_causatives,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    max_tokens=30,
    num_replacements=5)

## BNC SPOKEN

In [ ]:
dataset_path = "./syntactic-bootstrapping/corpora/BNC_SPOKEN/original/bnc_spoken.test.txt"   # input dataset (one sentence per line)
output_physical = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN/minimal_pairs_bnc_spoken_physical.txt"
output_mental   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN/minimal_pairs_bnc_spoken_mental.txt"
binned_dir = "./syntactic-bootstrapping/binned_data/BNC_SPOKEN"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_minimal_pairs_split(
    dataset_path,
    output_physical,
    output_mental,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    num_replacements=5)

In [ ]:
dataset_path = "./Desktop/syntactic-bootstrapping/corpora/BNC_SPOKEN_CANDOR/original/candor.test.txt"   # input dataset (one sentence per line)
output_causatives = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_causatives.txt"
output_non_causatives   = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_non_causatives.txt"
binned_dir = "./Desktop/syntactic-bootstrapping/binned_data/BNC_SPOKEN_CANDOR"

# Load bins and regex tokenizer
verb_bins = load_verb_bins(binned_dir)
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Generate minimal pairs
generate_min_pairs_caus_noncaus(
    dataset_path,
    output_causatives,
    output_non_causatives,
    verb_bins,
    pattern,
    spacy_model="en_core_web_sm",
    min_tokens=10,
    max_tokens=30,
    num_replacements=5)

## EVALUATE Mental verbs vs Physical Verbs

In [13]:
def evaluate_models_two_sets(base_model_names, seeds, physical_file_path, mental_file_path, limit_pairs=None):
    """
    Evaluate multiple models across seeds on two datasets of minimal pairs.

    Args:
        base_model_names (list): List of base model names (without seeds).
        seeds (list): List of seed integers.
        physical_file_path (str): Path to the physical verbs minimal pairs CSV file.
        mental_file_path (str): Path to the mental verbs minimal pairs CSV file.
        limit_pairs (int, optional): Limit number of sentence pairs for speed. Default None.

    Returns:
        dict: Average accuracies per base model for physical and mental sets.
    """

    model_names = []
    for base_model_name in base_model_names:
        for seed in seeds:
            model_names.append((base_model_name, f"{base_model_name}_{seed}"))


    def load_pairs(path):
        pairs = []
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                s1 = row.get("sentence1")
                s2 = row.get("sentence2")

                if not s1 or not s2:
                    continue

                s1 = s1.strip().strip('"').strip("'")
                s2 = s2.strip().strip('"').strip("'")

                pairs.append((s1, s2))

        print(f"✅ Loaded {len(pairs)} valid pairs from {path}")
        return pairs

    physical_pairs = load_pairs(physical_file_path)
    mental_pairs = load_pairs(mental_file_path)

    print(f"Loaded {len(physical_pairs)} physical pairs, {len(mental_pairs)} mental pairs")

    # Equalize dataset sizes
    min_size = min(len(physical_pairs), len(mental_pairs))
    if limit_pairs:
        min_size = min(min_size, limit_pairs)

    physical_pairs = physical_pairs[:min_size]
    mental_pairs = mental_pairs[:min_size]

    print(f"Truncated datasets to {min_size} pairs each for fair comparison.")

    def evaluate_model(model_name, pairs):
        scorer_model = scorer.IncrementalLMScorer(model_name, device="cuda")
        total = len(pairs)
        correct = 0
        for sent1, sent2 in pairs:
            if not sent1.strip() or not sent2.strip():
                continue
            try:
                score1 = scorer_model.sequence_score([sent1])[0]
                score2 = scorer_model.sequence_score([sent2])[0]
            except Exception as e:
                print(f"🚨 Error evaluating {model_name}: {e}")
                continue
            if score1 > score2:
                correct += 1
        return correct / total if total > 0 else 0.0

    results = {base: {"physical": [], "mental": []} for base in base_model_names}

    for base_model_name, model_name in model_names:
        print(f"\nEvaluating model: {model_name}...")

        acc_phys = evaluate_model(model_name, physical_pairs)
        acc_ment = evaluate_model(model_name, mental_pairs)

        results[base_model_name]["physical"].append(acc_phys)
        results[base_model_name]["mental"].append(acc_ment)

        print(f"{model_name} accuracy (physical): {acc_phys:.4f}")
        print(f"{model_name} accuracy (mental):   {acc_ment:.4f}")

    averages = {}
    print("\n=== Average accuracies across seeds per base model ===")
    for base_model_name in base_model_names:
        phys_accs = results[base_model_name]["physical"]
        ment_accs = results[base_model_name]["mental"]

        avg_phys = mean(phys_accs) if phys_accs else 0.0
        avg_ment = mean(ment_accs) if ment_accs else 0.0

        averages[base_model_name] = {"physical": avg_phys, "mental": avg_ment}

        print(f"\n{base_model_name}:")
        print(f"  Physical verbs: {avg_phys:.4f}")
        print(f"  Mental verbs:   {avg_ment:.4f}")

    return averages


In [ ]:
base_models = [
    "fpadovani/cds_ori",
    "fpadovani/cds_word",
    "fpadovani/cds_shuffle_1gr",
    "fpadovani/cds_shuffle_np"
]

seeds = [13]

physical_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_physical_stanza.txt"
mental_path   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_mental_stanza.txt"

results = evaluate_models_two_sets(base_models, seeds, physical_path, mental_path, limit_pairs=100000)
print("\nFinal results:", results)

✅ Loaded 26096 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_physical_stanza.txt
✅ Loaded 40187 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_mental_stanza.txt
Loaded 26096 physical pairs, 40187 mental pairs
Truncated datasets to 26096 pairs each for fair comparison.

Evaluating model: fpadovani/cds_ori_13...
fpadovani/cds_ori_13 accuracy (physical): 0.8742
fpadovani/cds_ori_13 accuracy (mental):   0.8367

Evaluating model: fpadovani/cds_word_13...
fpadovani/cds_word_13 accuracy (physical): 0.8408
fpadovani/cds_word_13 accuracy (mental):   0.8007

Evaluating model: fpadovani/cds_shuffle_1gr_13...
fpadovani/cds_shuffle_1gr_13 accuracy (physical): 0.7791
fpadovani/cds_shuffle_1gr_13 accuracy (mental):   0.7679

Evaluating model: fpadovani/cds_shuffle_np_13...
fpadovani/cds_shuffle_np_13 accuracy (physical): 0.7863
fpadovani/cds_shuffl

In [ ]:
base_model_names = [
    "fpadovani/wiki_ori",
    "fpadovani/wiki_word",
    "fpadovani/wiki_shuffle_1gr",
    "fpadovani/wiki_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
physical_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_physical.txt"
mental_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_mental.txt"

results = evaluate_models_two_sets(base_model_names, seeds, physical_path, mental_path, limit_pairs=100000)
print("\nFinal results:", results)


✅ Loaded 19342 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_physical.txt
✅ Loaded 5188 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_mental.txt
Loaded 19342 physical pairs, 5188 mental pairs
Truncated datasets to 5188 pairs each for fair comparison.

Evaluating model: fpadovani/wiki_ori_13...
fpadovani/wiki_ori_13 accuracy (physical): 0.5798
fpadovani/wiki_ori_13 accuracy (mental):   0.6475

Evaluating model: fpadovani/wiki_word_13...
fpadovani/wiki_word_13 accuracy (physical): 0.5426
fpadovani/wiki_word_13 accuracy (mental):   0.6049

Evaluating model: fpadovani/wiki_shuffle_1gr_13...
fpadovani/wiki_shuffle_1gr_13 accuracy (physical): 0.4753
fpadovani/wiki_shuffle_1gr_13 accuracy (mental):   0.4996

Evaluating model: fpadovani/wiki_shuffle_np_13...
fpadovani/wiki_shuffle_np_13 accuracy (physical): 0.4998
fpadovani/wiki_shuffle_np_13 accuracy (menta

In [ ]:

base_model_names = [
    "fpadovani/bnc_ori",
    "fpadovani/bnc_word",
    "fpadovani/bnc_shuffle_1gr",
    "fpadovani/bnc_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
physical_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_physical.txt"
mental_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_mental.txt"

results = evaluate_models_two_sets(base_model_names, seeds, physical_path, mental_path, limit_pairs=100000)
print("\nFinal results:", results)


✅ Loaded 25389 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_physical.txt
✅ Loaded 15016 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_mental.txt
Loaded 25389 physical pairs, 15016 mental pairs
Truncated datasets to 15016 pairs each for fair comparison.

Evaluating model: fpadovani/bnc_ori_13...
fpadovani/bnc_ori_13 accuracy (physical): 0.5456
fpadovani/bnc_ori_13 accuracy (mental):   0.5370

Evaluating model: fpadovani/bnc_word_13...
fpadovani/bnc_word_13 accuracy (physical): 0.5055
fpadovani/bnc_word_13 accuracy (mental):   0.4861

Evaluating model: fpadovani/bnc_shuffle_1gr_13...
fpadovani/bnc_shuffle_1gr_13 accuracy (physical): 0.4779
fpadovani/bnc_shuffle_1gr_13 accuracy (mental):   0.4111

Evaluating model: fpadovani/bnc_shuffle_np_13...
fpadovani/bnc_shuffle_np_13 accuracy (physical): 0.4910
fpadovani/bnc_shuffle_np_13 accuracy (mental):   0.4206


In [ ]:
base_model_names = [
    "fpadovani/candor_ori",
    "fpadovani/candor_word",
    "fpadovani/candor_shuffle_1gr",
    "fpadovani/candor_shuffle_np"
]

# 2. Seeds to append
seeds = [13]

physical_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_physical.txt"
mental_path = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_mental.txt"

results = evaluate_models_two_sets(base_model_names, seeds, physical_path, mental_path, limit_pairs=100000)
print("\nFinal results:", results)

✅ Loaded 22965 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_physical.txt
✅ Loaded 31904 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_mental.txt
Loaded 22965 physical pairs, 31904 mental pairs
Truncated datasets to 22965 pairs each for fair comparison.

Evaluating model: fpadovani/candor_ori_13...


fpadovani/candor_ori_13 accuracy (physical): 0.6166
fpadovani/candor_ori_13 accuracy (mental):   0.6321

Evaluating model: fpadovani/candor_word_13...
fpadovani/candor_word_13 accuracy (physical): 0.6045
fpadovani/candor_word_13 accuracy (mental):   0.6259

Evaluating model: fpadovani/candor_shuffle_1gr_13...
fpadovani/candor_shuffle_1gr_13 accuracy (physical): 0.5639
fpadovani/candor_shuffle_1gr_13 accuracy (mental):   0.5678

Evaluating model: fpadovani/candor_shuffle_np_13...
fpadovani/candor_shuffle_np_13 accuracy (physical): 0.5572
fpadovani/candor_shuffle_np_13 accuracy (mental):   0.5585

=== Average accuracies across seeds per base model ===

fpadovani/candor_ori:
  Physical verbs: 0.6166
  Mental verbs:   0.6321

fpadovani/candor_word:
  Physical verbs: 0.6045
  Mental verbs:   0.6259

fpadovani/candor_shuffle_1gr:
  Physical verbs: 0.5639
  Mental verbs:   0.5678

fpadovani/candor_shuffle_np:
  Physical verbs: 0.5572
  Mental verbs:   0.5585

Final results: {'fpadovani/candor

## EVALUATE CAUSATIVE AND NON CAUSATIVE VERBS

In [ ]:
base_models = [
    "fpadovani/cds_ori",
    "fpadovani/cds_word",
    "fpadovani/cds_shuffle_1gr",
    "fpadovani/cds_shuffle_np"
]

seeds = [13]

caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_causatives.txt"
non_caus   = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_non_causatives.txt"

results = evaluate_models_two_sets(base_models, seeds, caus, non_caus, limit_pairs=100000)
print("\nFinal results:", results)

✅ Loaded 2455 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_causatives.txt
✅ Loaded 51162 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/CHILDES_stanza/min_pairs_childes_non_causatives.txt
Loaded 2455 physical pairs, 51162 mental pairs
Truncated datasets to 2455 pairs each for fair comparison.

Evaluating model: fpadovani/cds_ori_13...
fpadovani/cds_ori_13 accuracy (physical): 0.8253
fpadovani/cds_ori_13 accuracy (mental):   0.9075

Evaluating model: fpadovani/cds_word_13...
fpadovani/cds_word_13 accuracy (physical): 0.7377
fpadovani/cds_word_13 accuracy (mental):   0.8762

Evaluating model: fpadovani/cds_shuffle_1gr_13...
fpadovani/cds_shuffle_1gr_13 accuracy (physical): 0.7303
fpadovani/cds_shuffle_1gr_13 accuracy (mental):   0.8501

Evaluating model: fpadovani/cds_shuffle_np_13...
fpadovani/cds_shuffle_np_13 accuracy (physical): 0.7218
fpadovani/cds_shuffle_np_13

In [ ]:
base_model_names = [
    "fpadovani/wiki_ori",
    "fpadovani/wiki_word",
    "fpadovani/wiki_shuffle_1gr",
    "fpadovani/wiki_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_causatives.txt"
non_caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_non_causatives.txt"

results = evaluate_models_two_sets(base_model_names, seeds, caus, non_caus, limit_pairs=100000)
print("\nFinal results:", results)


✅ Loaded 8498 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_causatives.txt
✅ Loaded 9564 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/WIKI/minimal_pairs_wiki_non_causatives.txt
Loaded 8498 physical pairs, 9564 mental pairs
Truncated datasets to 8498 pairs each for fair comparison.

Evaluating model: fpadovani/wiki_ori_13...


fpadovani/wiki_ori_13 accuracy (physical): 0.5473
fpadovani/wiki_ori_13 accuracy (mental):   0.5948

Evaluating model: fpadovani/wiki_word_13...
fpadovani/wiki_word_13 accuracy (physical): 0.5132
fpadovani/wiki_word_13 accuracy (mental):   0.5627

Evaluating model: fpadovani/wiki_shuffle_1gr_13...
fpadovani/wiki_shuffle_1gr_13 accuracy (physical): 0.4398
fpadovani/wiki_shuffle_1gr_13 accuracy (mental):   0.4596

Evaluating model: fpadovani/wiki_shuffle_np_13...
fpadovani/wiki_shuffle_np_13 accuracy (physical): 0.4681
fpadovani/wiki_shuffle_np_13 accuracy (mental):   0.4842

=== Average accuracies across seeds per base model ===

fpadovani/wiki_ori:
  Physical verbs: 0.5473
  Mental verbs:   0.5948

fpadovani/wiki_word:
  Physical verbs: 0.5132
  Mental verbs:   0.5627

fpadovani/wiki_shuffle_1gr:
  Physical verbs: 0.4398
  Mental verbs:   0.4596

fpadovani/wiki_shuffle_np:
  Physical verbs: 0.4681
  Mental verbs:   0.4842

Final results: {'fpadovani/wiki_ori': {'physical': 0.5473052482

In [ ]:

base_model_names = [
    "fpadovani/bnc_ori",
    "fpadovani/bnc_word",
    "fpadovani/bnc_shuffle_1gr",
    "fpadovani/bnc_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_causatives.txt"
non_caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_non_causatives.txt"

results = evaluate_models_two_sets(base_model_names, seeds, caus, non_caus, limit_pairs=100000)
print("\nFinal results:", results)


✅ Loaded 8595 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_causatives.txt
✅ Loaded 22778 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC/minimal_pairs_bnc_non_causatives.txt
Loaded 8595 physical pairs, 22778 mental pairs
Truncated datasets to 8595 pairs each for fair comparison.

Evaluating model: fpadovani/bnc_ori_13...
fpadovani/bnc_ori_13 accuracy (physical): 0.4983
fpadovani/bnc_ori_13 accuracy (mental):   0.5280

Evaluating model: fpadovani/bnc_word_13...
fpadovani/bnc_word_13 accuracy (physical): 0.4663
fpadovani/bnc_word_13 accuracy (mental):   0.4913

Evaluating model: fpadovani/bnc_shuffle_1gr_13...
fpadovani/bnc_shuffle_1gr_13 accuracy (physical): 0.4487
fpadovani/bnc_shuffle_1gr_13 accuracy (mental):   0.4256

Evaluating model: fpadovani/bnc_shuffle_np_13...
fpadovani/bnc_shuffle_np_13 accuracy (physical): 0.4652
fpadovani/bnc_shuffle_np_13 accuracy (mental):   

In [ ]:
base_model_names = [
    "fpadovani/candor_ori",
    "fpadovani/candor_word",
    "fpadovani/candor_shuffle_1gr",
    "fpadovani/candor_shuffle_np"
]

# 2. Seeds to append
seeds = [13]

caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_causatives.txt"
non_caus = "./syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_non_causatives.txt"

results = evaluate_models_two_sets(base_model_names, seeds, caus, non_caus, limit_pairs=100000)
print("\nFinal results:", results)

✅ Loaded 2420 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_causatives.txt
✅ Loaded 40942 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor_non_causatives.txt
Loaded 2420 physical pairs, 40942 mental pairs
Truncated datasets to 2420 pairs each for fair comparison.

Evaluating model: fpadovani/candor_ori_13...
fpadovani/candor_ori_13 accuracy (physical): 0.6058
fpadovani/candor_ori_13 accuracy (mental):   0.6401

Evaluating model: fpadovani/candor_word_13...
fpadovani/candor_word_13 accuracy (physical): 0.5657
fpadovani/candor_word_13 accuracy (mental):   0.6033

Evaluating model: fpadovani/candor_shuffle_1gr_13...
fpadovani/candor_shuffle_1gr_13 accuracy (physical): 0.5533
fpadovani/candor_shuffle_1gr_13 accuracy (mental):   0.5831

Evaluating model: fpadovani/candor_shuffle_np_13...
fpadovani/candor_shuffle_np_13 accuracy (

## GENERATE MINIMAL PAIRS WITH NOUN FOCUS

In [22]:
punct_fix = re.compile(r"\s+([.,?!])")  # optional punctuation cleaning

def generate_minimal_pairs_spacy_noun(
    input_file, 
    output_file, 
    binned_words, 
    pattern=re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?"),
    spacy_model="en_core_web_sm",
    min_tokens=10,  
    max_tokens = 30,      # lowered for short sentences like CHILDES
    num_replacements=5,
    batch_size=400
):
    """
    Generate noun-based minimal pairs using SpaCy, ensuring replacements come from the same bin.
    """
    nlp = spacy.load(spacy_model, disable=["ner"])
    
    with open(input_file, "r", encoding="utf-8") as f, \
         open(output_file, "w", encoding="utf-8") as out_file:
        
        for doc in nlp.pipe(f, batch_size=batch_size):
            tokens = [t.text for t in doc]
            if len(tokens) < min_tokens or len(tokens) > max_tokens:
                continue

            noun_tokens = [t for t in doc if t.pos_ == "NOUN"]
            if not noun_tokens:
                continue

            # Choose target noun
            target = noun_tokens[0] if len(noun_tokens) == 1 else random.choice(noun_tokens)
            pos, tag = target.pos_, target.tag_

            # Find bin(s) containing the original noun
            candidate_bins = [
                key for key in binned_words.keys()
                if key[0] == pos and key[1] == tag and target.text.lower() in [w.lower() for w in binned_words[key]]
            ]
            if not candidate_bins:
                continue

            bin_key = random.choice(candidate_bins)

            # Sample replacements from same bin, excluding original
            candidates = [w for w in binned_words[bin_key] if w.lower() != target.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Original sentence
            original_sent = " ".join([t.text for t in doc])
            original_sent = punct_fix.sub(r"\1", original_sent)

            # Generate minimal pairs
            token_texts = [t.text for t in doc]
            for rep in replacements:
                new_tokens = token_texts.copy()
                new_tokens[target.i] = rep
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                out_file.write(f"{original_sent.strip()},{new_sent.strip()}\n")
    
    print(f"SpaCy noun-based minimal pairs saved to {output_file}")

In [23]:
import stanza

punct_fix = re.compile(r"\s+([.,?!])")  # optional punctuation cleaning

def generate_minimal_pairs_stanza_noun(
    input_file, 
    output_file, 
    binned_words, 
    pos_model_path, 
    parser_model_path,
    pattern=re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?"),
    min_tokens=10,
    num_replacements=5,
    use_gpu=True
):
    """
    Generate noun-based minimal pairs using Stanza, ensuring replacements come from the same bin.
    """
    nlp = stanza.Pipeline(
        lang='en',
        processors='tokenize,pos,lemma,depparse',
        use_gpu=use_gpu,
        pos_model_path=pos_model_path,
        depparse_model_path=parser_model_path,
        tokenize_no_ssplit=True
    )

    with open(input_file, "r", encoding="utf-8") as f, \
         open(output_file, "w", encoding="utf-8") as out_file:

        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens_in_line = pattern.findall(line)
            if len(tokens_in_line) < min_tokens:
                continue

            doc = nlp(line)
            all_words = [w for sent in doc.sentences for w in sent.words]

            # Select nouns
            noun_tokens = [w for w in all_words if w.upos == "NOUN"]
            if not noun_tokens:
                continue

            target = noun_tokens[0] if len(noun_tokens) == 1 else random.choice(noun_tokens)
            pos, tag = target.upos, target.xpos

            # Find bin(s) containing the original noun
            candidate_bins = [
                key for key in binned_words.keys()
                if key[0] == pos and key[1] == tag and target.text.lower() in [v.lower() for v in binned_words[key]]
            ]
            if not candidate_bins:
                continue

            bin_key = random.choice(candidate_bins)

            # Sample replacements from same bin, excluding original noun
            candidates = [v for v in binned_words[bin_key] if v.lower() != target.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Original sentence
            original_sent = " ".join([w.text for w in all_words])
            original_sent = punct_fix.sub(r"\1", original_sent)

            # Generate minimal pairs
            for rep in replacements:
                new_tokens = [w.text for w in all_words]
                new_tokens[target.id - 1] = rep  # Stanza IDs are 1-based
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)
                out_file.write(f"{original_sent.strip()},{new_sent.strip()}\n")

    print(f"Stanza noun-based minimal pairs saved to {output_file}")

## CHILDES Spacy

In [ ]:
binned_data_dir = "./syntactic-bootstrapping/binned_data/CHILDES_spacy"
verb_bins = load_verb_bins(binned_data_dir)


dataset_path = "./syntactic-bootstrapping/corpora/CHILDES/original/childes.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/CHILDES_spacy/minimal_pairs_childes_spacy.txt"

pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

generate_minimal_pairs_spacy_noun(dataset_path, output_txt, verb_bins, pattern)

Loaded bins for 500 POS-TAG-bin combinations.
SpaCy noun-based minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/CHILDES_spacy/minimal_pairs_childes_spacy.txt


## CHILDES Stanza 

In [ ]:
import stanza
pos_tagger_model = './syntactic-bootstrapping/saved_models/pos/en_childes_charlm_tagger.pt'
parser_model = './syntactic-bootstrapping/saved_models/depparse/en_childes_charlm_parser.pt'

binned_data_dir = "./syntactic-bootstrapping/binned_data/CHILDES_stanza"
verb_bins = load_verb_bins(binned_data_dir)


dataset_path = "./syntactic-bootstrapping/corpora/CHILDES/original/childes.test.txt"
output_txt = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/CHILDES_stanza/minimal_pairs_childes_stanza.txt"


generate_minimal_pairs_stanza_noun(dataset_path, output_txt, verb_bins, pos_model_path=pos_tagger_model, parser_model_path=parser_model)

2025-09-23 15:05:53 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Loaded bins for 470 POS-TAG-bin combinations.


2025-09-23 15:05:53 INFO: Downloaded file to /home/p318482/stanza_resources/resources.json
2025-09-23 15:05:53 WARNING: Language en package default expects mwt, which has been added
2025-09-23 15:05:54 INFO: Loading these models for language: en (English):
| Processor | Package                 |
---------------------------------------
| tokenize  | combined                |
| mwt       | combined                |
| pos       | /home/p318..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | /home/p318..._parser.pt |

2025-09-23 15:05:54 INFO: Using device: cuda
2025-09-23 15:05:54 INFO: Loading: tokenize
2025-09-23 15:05:54 INFO: Loading: mwt
2025-09-23 15:05:54 INFO: Loading: pos
2025-09-23 15:05:57 INFO: Loading: lemma
2025-09-23 15:05:58 INFO: Loading: depparse
2025-09-23 15:06:00 INFO: Done loading processors!


Stanza noun-based minimal pairs saved to /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/CHILDES_stanza/minimal_pairs_childes_stanza.txt


## WIKI

In [ ]:
binned_data_dir = './Desktop/syntactic-bootstrapping/binned_data/WIKI'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./Desktop/syntactic-bootstrapping/corpora/WIKI/original/wiki.test.txt"
output_txt = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_wiki.txt"

generate_minimal_pairs_spacy_noun(dataset_path, output_txt, verb_bins, pattern)


## BNC 

In [ ]:
binned_data_dir = './Desktop/syntactic-bootstrapping/binned_data/BNC'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./Desktop/syntactic-bootstrapping/corpora/BNC/original/bnc.test.txt"
output_txt = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_bnc.txt"

generate_minimal_pairs_spacy_noun(dataset_path, output_txt, verb_bins, pattern)


## BNC_SPOKEN

In [ ]:
binned_data_dir = './Desktop/syntactic-bootstrapping/binned_data/BNC_SPOKEN'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./Desktop/syntactic-bootstrapping/corpora/BNC_SPOKEN/original/bnc_spoken.test.txt"
output_txt = "./Desktop/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_bnc_spoken.txt"

generate_minimal_pairs_spacy_noun(dataset_path, output_txt, verb_bins, pattern)


## Evaluate models on Noun minimal pairs

## CHILDES

In [ ]:
base_model_names = [
    "fpadovani/cds_shuffle_np",
    "fpadovani/cds_shuffle_np"
]


# 2. Seeds to append
seeds = [13]
input_data_file = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_childes_stanza.txt"
evaluate_models(base_model_names, seeds, input_data_file, limit_pairs=100000)

✅ Loaded 178332 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_childes_stanza.txt
Loaded 100000 minimal pairs for evaluation.

Evaluating model: fpadovani/cds_shuffle_np_13...
fpadovani/cds_shuffle_np_13 accuracy: 0.8081

Evaluating model: fpadovani/cds_shuffle_np_13...
fpadovani/cds_shuffle_np_13 accuracy: 0.8081

=== Average accuracies across seeds per model ===
fpadovani/cds_shuffle_np: Average accuracy: 0.8081
fpadovani/cds_shuffle_np: Average accuracy: 0.8081


{'fpadovani/cds_shuffle_np': 0.80811}

## WIKIPEDIA

In [ ]:
base_model_names = [
    "fpadovani/wiki_ori",
    "fpadovani/wiki_word_noun",
    "fpadovani/wiki_shuffle_1gr",
    "fpadovani/wiki_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
input_data_file = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_wiki.txt"
evaluate_models(base_model_names, seeds, input_data_file, limit_pairs=100000)


✅ Loaded 351211 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_wiki.txt
Loaded 100000 minimal pairs for evaluation.

Evaluating model: fpadovani/wiki_ori_13...
⚠️ Skipping empty sentence pair #32916
⚠️ Skipping empty sentence pair #32917
⚠️ Skipping empty sentence pair #32918
⚠️ Skipping empty sentence pair #32919
⚠️ Skipping empty sentence pair #32920
⚠️ Skipping empty sentence pair #50778
⚠️ Skipping empty sentence pair #50779
⚠️ Skipping empty sentence pair #50780
⚠️ Skipping empty sentence pair #50781
⚠️ Skipping empty sentence pair #50782
fpadovani/wiki_ori_13 accuracy: 0.6101

Evaluating model: fpadovani/wiki_word_noun_13...
⚠️ Skipping empty sentence pair #32916
⚠️ Skipping empty sentence pair #32917
⚠️ Skipping empty sentence pair #32918
⚠️ Skipping empty sentence pair #32919
⚠️ Skipping empty sentence pair #32920
⚠️ Skipping empty sentence pair #50778
⚠️ Skipping empty sentence pair #50779
⚠️ Skipping empty sentence pai

{'fpadovani/wiki_ori': 0.61012,
 'fpadovani/wiki_word_noun': 0.5026,
 'fpadovani/wiki_shuffle_1gr': 0.54176,
 'fpadovani/wiki_shuffle_np': 0.57736}

## BNC

In [ ]:
base_model_names = [
    "fpadovani/bnc_ori",
    "fpadovani/bnc_word"
]

# 2. Seeds to append
seeds = [13]
input_data_file = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_bnc.txt"
evaluate_models(base_model_names, seeds, input_data_file, limit_pairs=100000)

✅ Loaded 361775 valid pairs from /home/p318482/syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_bnc.txt
Loaded 100000 minimal pairs for evaluation.

Evaluating model: fpadovani/bnc_shuffle_1gr_13...
fpadovani/bnc_shuffle_1gr_13 accuracy: 0.5054

Evaluating model: fpadovani/bnc_shuffle_np_13...
fpadovani/bnc_shuffle_np_13 accuracy: 0.5317

=== Average accuracies across seeds per model ===
fpadovani/bnc_shuffle_1gr: Average accuracy: 0.5054
fpadovani/bnc_shuffle_np: Average accuracy: 0.5317


{'fpadovani/bnc_shuffle_1gr': 0.50542, 'fpadovani/bnc_shuffle_np': 0.5317}

## BNC_SPOKEN_CANDOR

In [ ]:
base_model_names = [
    "fpadovani/candor_ori",
    "fpadovani/candor_word_noun",
    "fpadovani/candor_shuffle_1gr",
    "fpadovani/candor_shuffle_np"
]

# 2. Seeds to append
seeds = [13]
input_data_file = "./syntactic-bootstrapping/evaluation/clm/min_pairs/noun_focus/minimal_pairs_candor.txt"
evaluate_models(base_model_names, seeds, input_data_file, limit_pairs=100000)